# Employee Attrition Prediction: Feature Engineering & Data Preprocessing

### 3.1. Objective

The objective of this phase is to transform the cleaned dataset from Phase 1 into a machine-learning-ready dataset for employee attrition prediction.

The preprocessing workflow will includes:
- Defining the prediction target
- Removing irrelevant identifier and constant variables
- Separating predictors from the target variable
- Splitting the data into training and testing sets
- Identifying numerical and categorical features
- Encoding categorical variables
- Scaling numerical variables where appropriate
- Preventing data leakage through a training-only preprocessing workflow
- Preparing the final datasets for model development
- Performing appropriate feature selection where necessary

A key principle in this phase is that information from the test set must not influence preprocessing decisions or parameter estimation. Therefore, the train/test split will occur before fitting preprocessing transformations.

#### 3.2. Import Libraries

In [29]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

In [30]:
# Load Cleaned Dataset
df = pd.read_csv("clean_employee_attrition.csv")

In [31]:
# To Verify the dataset

print("Dataset Shape:", df.shape)

display(df.head())

display(df.info())

print("\nMissing Values:")
display(df.isnull().sum().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Dataset Shape: (1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

None


Missing Values:


np.int64(0)


Duplicate Rows: 0


#### 3.3. To Confirm the Target Variable

The target variable for this classification problem is `Attrition`.

It represents whether an employee left the organization:

- `Yes` = Employee left
- `No` = Employee stayed

For machine learning, the target will subsequently be converted into a binary numerical representation.

In [32]:
df['Attrition'].value_counts()

Attrition
No     1233
Yes     237
Name: count, dtype: int64

In [33]:
df['Attrition'].value_counts(normalize=True).mul(100).round(2)

Attrition
No     83.88
Yes    16.12
Name: proportion, dtype: float64

This confirms the class distribution.

In [34]:
# To Convert target to binary in order to simplify machine learning tasks, improve model accuracy, and match algorithm

df['Attrition_Binary'] = df['Attrition'].map({'No': 0,'Yes': 1})

In [35]:
# to confirm it

df[['Attrition', 'Attrition_Binary']].drop_duplicates()

,Attrition,Attrition_Binary
0,Yes,1
1,No,0


#### 3.4.1. Hence, we remove the original target so as not to have both: Attrition and Attrition_Binary

In [36]:
X = df.drop(columns=['Attrition', 'Attrition_Binary'])
y = df['Attrition_Binary']

# Then check

print("Features shape:", X.shape)
print("Target shape:", y.shape)

Features shape: (1470, 34)
Target shape: (1470,)


#### 3.4.2. Removing Irrelevant and Identifier Variables

Several variables will be excluded from the predictive feature set because they do not represent meaningful employee characteristics for attrition prediction.

`EmployeeNumber` is an identifier and does not represent an underlying employee attribute. Including arbitrary identifiers may introduce noise and reduce model interpretability.

`EmployeeCount`, `StandardHours`, and `Over18` contain constant or effectively non-informative information in this dataset and therefore provide no useful predictive variation.

These variables will be removed before model preprocessing.

In [37]:
irrelevant_cols = ['EmployeeNumber','EmployeeCount','StandardHours','Over18']

X = X.drop(columns=irrelevant_cols)

print("Features after removing irrelevant columns:", X.shape)

Features after removing irrelevant columns: (1470, 30)


In [38]:
X.columns.tolist()

['Age',
 'BusinessTravel',
 'DailyRate',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EnvironmentSatisfaction',
 'Gender',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

### 3.5. Train/Test Split

The dataset will be divided into training and testing subsets before preprocessing.

The training set will be used to fit preprocessing transformations and machine learning models, while the test set will remain unseen until model evaluation.

This prevents data leakage, where information from the test set could influence preprocessing parameters or model development.

A stratified split is used to preserve the proportion of employees who left and stayed in both subsets

In [39]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=42,stratify=y)

In [40]:
# To Verify the split

print("Training features:", X_train.shape)
print("Testing features:", X_test.shape)

print("\nTraining target distribution:")
display(y_train.value_counts(normalize=True).mul(100).round(2))

print("\nTesting target distribution:")
display(y_test.value_counts(normalize=True).mul(100).round(2))

Training features: (1176, 30)
Testing features: (294, 30)

Training target distribution:


Attrition_Binary
0    83.84
1    16.16
Name: proportion, dtype: float64


Testing target distribution:


Attrition_Binary
0    84.01
1    15.99
Name: proportion, dtype: float64

### 3.6. Identifing numerical and categorical features

In [41]:
# To reinforces our leakage-safe workflow, we use 'X_train' rather than the entire X

numerical_features = X_train.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

categorical_features = X_train.select_dtypes(
    include=['object']
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

Numerical Features:
['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']

Categorical Features:
['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']


In [42]:
# Hence a santity check
print("Number of numerical features:", len(numerical_features))
print("Number of categorical features:", len(categorical_features))
print("Total features before encoding:", len(X_train.columns))

Number of numerical features: 23
Number of categorical features: 7
Total features before encoding: 30


### 3.7. Build the preprocessing pipeline

#### 3.7.1. Feature Preprocessing

The training and testing datasets contain both numerical and categorical variables, which require different preprocessing techniques.

Numerical variables will be standardized using `StandardScaler` so that variables measured on different scales are placed on a comparable scale, while the Categorical variables will be converted into numerical representations using one-hot encoding.

A `ColumnTransformer` will be used to apply the appropriate transformation to each feature type.

However, the preprocessing transformations will be fitted only on the training data and subsequently applied to the test data. This prevents information from the test set from influencing the preprocessing process and helps prevent data leakage.

#### 3.7.2.Create the preprocessing transformer

We will use **drop='first'** since a categorical variable such as:
Gender:
Female
Male
we don't need both columns after one-hot encoding to avoid unnecessary redundancy among dummy variables.

Also, **handle_unknown='ignore'** will be introduced as well so that if an unseen category appears in the test set, the transformer will not crash.

In [43]:
preprocessor = ColumnTransformer(transformers=[('num',StandardScaler(),numerical_features),('cat',OneHotEncoder(handle_unknown='ignore',
                drop='first'),categorical_features)])

### 3.8. To fit Preprocessing on Training Data

The preprocessing transformer is fitted exclusively on the training dataset. The test dataset is not used to calculate scaling parameters or determine categorical encoding. This ensures that the test set remains unseen during preprocessing and prevents data leakage.

In [44]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

### 3.8.1. Saving the Preprocessing Pipeline

To ensure reproducibility and consistency across the different stages of the project, the fitted preprocessing pipeline is saved using `joblib`.

The preprocessing pipeline contains the transformations applied to the machine-learning features, including numerical scaling and categorical encoding. It was fitted exclusively on the training data and then used to transform both the training and test datasets.

Saving the fitted preprocessor allows the exact same transformations to be reused during model evaluation in Phase 5 without fitting the preprocessing steps again. This helps maintain consistency between the data used during model development and the unseen test data used for final evaluation.

The saved preprocessing object will be loaded in subsequent phases rather than recreating the preprocessing process from scratch.

In [45]:
import joblib

joblib.dump(preprocessor,"phase3_preprocessor.pkl")

print("Preprocessor saved successfully.")

Preprocessor saved successfully.


In [46]:
# Hence, we Check the processed data

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (1176, 44)
Processed testing shape: (294, 44)


### 3.9. To Get the feature names

In [47]:
feature_names = preprocessor.get_feature_names_out()

print("Number of processed features:", len(feature_names))

feature_names

Number of processed features: 44


array(['num__Age', 'num__DailyRate', 'num__DistanceFromHome',
       'num__Education', 'num__EnvironmentSatisfaction',
       'num__HourlyRate', 'num__JobInvolvement', 'num__JobLevel',
       'num__JobSatisfaction', 'num__MonthlyIncome', 'num__MonthlyRate',
       'num__NumCompaniesWorked', 'num__PercentSalaryHike',
       'num__PerformanceRating', 'num__RelationshipSatisfaction',
       'num__StockOptionLevel', 'num__TotalWorkingYears',
       'num__TrainingTimesLastYear', 'num__WorkLifeBalance',
       'num__YearsAtCompany', 'num__YearsInCurrentRole',
       'num__YearsSinceLastPromotion', 'num__YearsWithCurrManager',
       'cat__BusinessTravel_Travel_Frequently',
       'cat__BusinessTravel_Travel_Rarely',
       'cat__Department_Research & Development', 'cat__Department_Sales',
       'cat__EducationField_Life Sciences',
       'cat__EducationField_Marketing', 'cat__EducationField_Medical',
       'cat__EducationField_Other',
       'cat__EducationField_Technical Degree', 'cat__Ge

### 3.10. Converting the processed data back to DataFrames

In [48]:
X_train_processed = pd.DataFrame(X_train_processed.toarray()
    if hasattr(X_train_processed, "toarray")
    else X_train_processed,columns=feature_names,index=X_train.index)

X_test_processed = pd.DataFrame(X_test_processed.toarray()
    if hasattr(X_test_processed, "toarray")
    else X_test_processed,columns=feature_names,index=X_test.index)

In [49]:
# to confirm 

display(X_train_processed.head())

,num__Age,num__DailyRate,num__DistanceFromHome,num__Education,num__EnvironmentSatisfaction,num__HourlyRate,num__JobInvolvement,num__JobLevel,num__JobSatisfaction,num__MonthlyIncome,...,cat__JobRole_Laboratory Technician,cat__JobRole_Manager,cat__JobRole_Manufacturing Director,cat__JobRole_Research Director,cat__JobRole_Research Scientist,cat__JobRole_Sales Executive,cat__JobRole_Sales Representative,cat__MaritalStatus_Married,cat__MaritalStatus_Single,cat__OverTime_Yes
1194,1.090194,1.049455,-0.899915,1.064209,-0.658710,-0.908436,1.795282,1.762189,-0.647997,2.026752,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
128,-1.634828,-0.523449,-0.899915,-1.855332,0.260202,1.694111,0.373564,-0.986265,1.153526,-0.864408,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
810,0.981193,-0.992080,-0.777610,-1.855332,-1.577622,-0.662913,0.373564,1.762189,0.252765,2.347706,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
478,-1.307825,-0.453653,0.445433,-1.855332,-0.658710,-1.252169,0.373564,-0.986265,0.252765,-0.956202,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
491,0.654191,0.491086,-0.043784,2.037390,1.179114,0.319180,0.373564,-0.070114,0.252765,-0.185956,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [50]:
# to confirm 

display(X_test_processed.head())

,num__Age,num__DailyRate,num__DistanceFromHome,num__Education,num__EnvironmentSatisfaction,num__HourlyRate,num__JobInvolvement,num__JobLevel,num__JobSatisfaction,num__MonthlyIncome,...,cat__JobRole_Laboratory Technician,cat__JobRole_Manager,cat__JobRole_Manufacturing Director,cat__JobRole_Research Director,cat__JobRole_Research Scientist,cat__JobRole_Sales Executive,cat__JobRole_Sales Representative,cat__MaritalStatus_Married,cat__MaritalStatus_Single,cat__OverTime_Yes
1061,-1.416826,0.064832,0.445433,-0.882152,1.179114,0.613808,0.373564,-0.986265,-0.647997,-0.969745,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0
891,0.763191,0.780241,-0.899915,-1.855332,-1.577622,0.319180,1.795282,-0.986265,1.153526,-0.974474,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
456,-0.653820,-0.289134,-0.288393,0.091029,0.260202,-1.055750,-1.048155,0.846038,1.153526,1.077650,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
922,0.763191,0.984644,-0.655306,-0.882152,0.260202,1.301274,1.795282,2.678340,-1.548758,2.718533,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
69,-0.108815,-1.211439,-0.043784,0.091029,1.179114,0.662913,-1.048155,-0.986265,0.252765,-0.678457,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0


### 3.11. A santity test to verify that there is no leakage

In [51]:
print("Training rows:", X_train_processed.shape[0])
print("Testing rows:", X_test_processed.shape[0])

print("Training features:", X_train_processed.shape[1])
print("Testing features:", X_test_processed.shape[1])

print("\nMissing values in processed training data:",
      X_train_processed.isnull().sum().sum())

print("Missing values in processed testing data:",
      X_test_processed.isnull().sum().sum())

Training rows: 1176
Testing rows: 294
Training features: 44
Testing features: 44

Missing values in processed training data: 0
Missing values in processed testing data: 0


### 3.12. Preprocessing Summary

The preprocessing pipeline successfully transformed the dataset into a machine-learning-ready format.

- Numerical variables were standardized using `StandardScaler`.
- Categorical variables were transformed using one-hot encoding.
- The first category of each categorical variable was dropped to avoid redundant dummy variables.
- Unknown categories are handled safely using `handle_unknown='ignore'`.
- The preprocessing transformer was fitted only on the training dataset.
- The fitted transformer was then used to transform both the training and testing datasets.
- The same processed feature structure is maintained across both datasets.

This workflow minimizes the risk of data leakage and ensures that the testing data remains unseen during preprocessing.

### 3.13. Preparing the target variables

To make sure the target is aligned with the processed feature sets.

In [52]:
y_train = y_train.copy()
y_test = y_test.copy()

print("Training target:", y_train.shape)
print("Testing target:", y_test.shape)

Training target: (1176,)
Testing target: (294,)


In [53]:
print("Training target distribution:")
display(y_train.value_counts())

print("\nTesting target distribution:")
display(y_test.value_counts())

Training target distribution:


Attrition_Binary
0    986
1    190
Name: count, dtype: int64


Testing target distribution:


Attrition_Binary
0    247
1     47
Name: count, dtype: int64

### 3.14. The feature-selection decision

**Feature Selection Strategy**

- No additional features were removed solely on the basis of individual statistical significance or correlation during preprocessing.
- Although the EDA identified variables that were more strongly associated with attrition, a weak individual association does not necessarily mean that a variable has no predictive value when combined with other features.
- Therefore, the initial modelling dataset retains the available cleaned features after removing identifiers and constant/non-informative variables.

Feature importance and model-based feature selection will be evaluated during the model development and evaluation phases.

### 3.15. ML-ready final check

In [54]:
print("\033[4m FINAL DATASET CHECK \033[0m")

print(f"Training features: {X_train_processed.shape}")
print(f"Testing features:  {X_test_processed.shape}")

print(f"Training target:   {y_train.shape}")
print(f"Testing target:    {y_test.shape}")

print(f"\nTraining missing values: {X_train_processed.isnull().sum().sum()}")
print(f"Testing missing values:  {X_test_processed.isnull().sum().sum()}")

print("\nPreprocessing complete.")

 FINAL DATASET CHECK 
Training features: (1176, 44)
Testing features:  (294, 44)
Training target:   (1176,)
Testing target:    (294,)

Training missing values: 0
Testing missing values:  0

Preprocessing complete.


### 3.15 Conclusion

Feature engineering and preprocessing have been completed using a leakage-aware workflow.

The cleaned dataset was separated into predictor variables and a binary attrition target. Identifier and non-informative constant variables were removed before modelling. The dataset was then divided into training and testing subsets using a stratified 80/20 split to preserve the distribution of the target variable. Numerical features were standardized, while categorical features were transformed using one-hot encoding. These transformations were fitted exclusively on the training dataset and subsequently applied to the test dataset, preventing test-set information from influencing preprocessing.

The resulting datasets are now machine-learning-ready and can be used for classification model development.

The next phase will involve training and comparing multiple classification algorithms, including Logistic Regression, Decision Tree and Random Forest models.